In [0]:
dbutils.widgets.text("catalog", "fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import col, date_format, expr, weekofyear, dayofweek, lit
from pyspark.sql.types import IntegerType

# Build dim_date from 2026-03-01 to 2026-03-31
start_date = "2026-03-01"
end_date = "2026-03-31"

dim_date = (
    spark.sql(f"SELECT sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day) as d_array")
    .select(expr("explode(d_array)").alias("calendar_date"))
    .select(
        date_format(col("calendar_date"), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
        col("calendar_date").cast("date").alias("calendar_date"),
        date_format(col("calendar_date"), "EEEE").alias("day_name"),
        weekofyear(col("calendar_date")).cast(IntegerType()).alias("iso_week"),
        (dayofweek(col("calendar_date")).isin(1, 7)).alias("is_weekend"),
    )
)

(dim_date.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_date"))

display(spark.table(f"{catalog}.gold.dim_date").orderBy("date_key"))

date_key,calendar_date,day_name,iso_week,is_weekend
20260301,2026-03-01,Sunday,9,true
20260302,2026-03-02,Monday,10,false
20260303,2026-03-03,Tuesday,10,false
20260304,2026-03-04,Wednesday,10,false
20260305,2026-03-05,Thursday,10,false
20260306,2026-03-06,Friday,10,false
20260307,2026-03-07,Saturday,10,true
20260308,2026-03-08,Sunday,10,true
20260309,2026-03-09,Monday,11,false
20260310,2026-03-10,Tuesday,11,false


In [0]:
from pyspark.sql.functions import *

dim_store = (
    spark.read.table(f"{catalog}.silver.stores")
    .select(
        col("store_id"),
        col("store_name"),
        col("city"),
        col("state"),
        col("region"),
        col("opening_date"),
    )
)

(dim_store.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_store"))

In [0]:
from pyspark.sql.functions import *

dim_products = (
    spark.read.table(f"{catalog}.silver.products")
    .select(
        col("product_id"),
        col("product_name"),
        col("brand"),
        col("category"),
        col("pack"),
        col("mrp"),
        col("selling_price"),
        col("is_active"),
        col("snapshot_date"),
    )
    .dropDuplicates(["product_id"])
)

(dim_products.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_products"))

In [0]:
from pyspark.sql.functions import *

FROM_COL = "effective_from"
TO_COL = "effective_to"

dim_customers = (
    spark.read.table(f"{catalog}.silver.customers_scd_t2")
    .withColumn("customer_sk", monotonically_increasing_id())
    .select(
        col("customer_sk").cast("bigint"),
        col("customer_id").cast("string"),
        col("full_name"),
        col("email"),
        col("phone"),
        col("city"),
        col("tier"),
        col(FROM_COL).alias("effective_from"),
        col(TO_COL).alias("effective_to"),
    )
)

(
    dim_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_customers")
)

In [0]:
from pyspark.sql.functions import *

DIM_TABLE = "gold.YOUR_DIM_TABLE"
FROM_COL = "valid_from"
TO_COL = "valid_to"

orders = spark.read.table(f"{catalog}.silver.orders")
items = spark.read.table(f"{catalog}.silver.order_items")

dim_customer = (
    spark.read.table(f"{catalog}.gold.dim_customers")
    .select(
        col("customer_sk"),
        col("customer_id").cast("string").alias("dc_customer_id"),
        col("effective_from").cast("date").alias("eff_from"),
        coalesce(col("effective_to").cast("date"), lit("9999-12-31").cast("date")).alias("eff_to"),
    )
)

line_agg = (
    items.groupBy("order_id")
    .agg(
        count("*").cast("int").alias("line_count"),
        sum("quantity").cast("int").alias("units"),
        sum("gross_amount").cast("decimal(12,2)").alias("gross_amount"),
        sum("discount_amount").cast("decimal(12,2)").alias("discount_amount"),
        sum("net_amount").cast("decimal(12,2)").alias("net_item_amount"),
    )
)

fact_orders = (
    orders
    .join(dim_customer,
          (orders.customer_id == dim_customer.dc_customer_id) &
          (orders.order_date >= dim_customer.eff_from) &
          (orders.order_date < dim_customer.eff_to),
          "left")
    .join(line_agg, "order_id", "left")
    .withColumn("order_date_key", date_format("order_date", "yyyyMMdd").cast("int"))
    .withColumn("net_item_amount", coalesce("net_item_amount", lit(0)).cast("decimal(12,2)"))
    .withColumn("order_total", (col("net_item_amount") + coalesce("delivery_fee", lit(0))).cast("decimal(12,2)"))
    .withColumn("is_on_time",
                when(col("order_status") == "DELIVERED",
                     coalesce(col("actual_mins") <= col("promised_mins"), lit(False))))
    .select(
        col("order_id"),
        col("order_date_key"),
        col("customer_sk").cast("bigint"),
        col("customer_id"),
        col("store_id"),
        col("order_status"),
        col("payment_method"),
        col("channel"),
        coalesce("line_count", lit(0)).cast("int").alias("line_count"),
        coalesce("units", lit(0)).cast("int").alias("units"),
        coalesce("gross_amount", lit(0)).cast("decimal(12,2)").alias("gross_amount"),
        coalesce("discount_amount", lit(0)).cast("decimal(12,2)").alias("discount_amount"),
        col("net_item_amount"),
        col("delivery_fee"),
        col("order_total"),
        col("promised_mins"),
        col("actual_mins"),
        col("is_on_time"),
    )
)

(fact_orders.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.fact_orders"))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit, coalesce

from pyspark.sql import Window

def duplicate_order_ids():
    df = spark.read.table(f"{catalog}.bronze.orders")
    window = Window.partitionBy("order_id").orderBy(desc("updated_at"))
    df = df.withColumn("rn", row_number().over(window))
    df = df.filter(col("rn") == 1).drop("rn")
    return df

latest_df = duplicate_order_ids().withColumn("order_id", col("order_id").cast("string"))

quarantine_ids = (
    spark.read.table(f"{catalog}.silver.quarantine_orders")
    .select(col("order_id").cast("string").alias("order_id"))
    .distinct()
)

df = latest_df.join(quarantine_ids, "order_id", "left_anti")

df = (
    df.select("*", F.posexplode("items").alias("pos", "item"))
    .drop("items")
    .withColumn("line_number", (col("pos") + 1).cast("int"))
)

df = (
    df.withColumn("product_id", col("item.product_id").cast("string"))
    .withColumn("quantity", when(col("item.qty") > 0, col("item.qty")).otherwise(lit(None)))
    .withColumn("unit_price", when(col("item.unit_price") > 0, col("item.unit_price")).otherwise(lit(None)))
    .withColumn("discount_amount", when(col("item.discount") >= 0, col("item.discount")).otherwise(lit(None)))
    .withColumn("gross_amount", col("quantity") * col("unit_price"))
    .withColumn("net_amount", col("gross_amount") - coalesce(col("discount_amount"), lit(0)))
)

df = df.select(
    "order_id", "line_number", "product_id", "quantity",
    "unit_price", "discount_amount", "gross_amount", "net_amount"
)

(
    df.write
    .option("mergeSchema", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.order_items")
)

In [0]:
from pyspark.sql.functions import *

fact = (
    spark.read.table(f"{catalog}.gold.fact_orders")
    .withColumn("order_date", to_date(col("order_date_key").cast("string"), "yyyyMMdd"))
)

stores = (
    spark.read.table(f"{catalog}.gold.dim_store")
    .select(col("store_id").cast("string").alias("store_id"), col("city").cast("string").alias("city"))
)

bounds = fact.agg(min("order_date").alias("mn"), max("order_date").alias("mx")).first()

dates = spark.range(1).select(
    explode(sequence(lit(bounds["mn"]), lit(bounds["mx"]))).alias("order_date")
)

grid = dates.crossJoin(stores)

delivered = col("order_status") == "DELIVERED"

daily = (
    fact.groupBy("order_date", "store_id")
    .agg(
        count(when(delivered, 1)).alias("delivered_orders"),
        count(when(col("order_status") == "CANCELLED", 1)).alias("cancelled_orders"),
        sum(when(delivered, col("units"))).alias("units_sold"),
        sum(when(delivered, col("gross_amount"))).alias("gross_amount"),
        sum(when(delivered, col("discount_amount"))).alias("discount_amount"),
        sum(when(delivered, col("net_item_amount"))).alias("net_item_revenue"),
        sum(when(delivered, col("delivery_fee"))).alias("delivery_fee_revenue"),
        count(when(delivered & (col("is_on_time") == True), 1)).alias("on_time_orders"),
    )
)

agg_daily_store_sales = (
    grid.join(daily, ["order_date", "store_id"], "left")
    .withColumn("delivered_orders", coalesce("delivered_orders", lit(0)).cast("int"))
    .withColumn("cancelled_orders", coalesce("cancelled_orders", lit(0)).cast("int"))
    .withColumn("units_sold", coalesce("units_sold", lit(0)).cast("int"))
    .withColumn("gross_amount", coalesce("gross_amount", lit(0)).cast("decimal(12,2)"))
    .withColumn("discount_amount", coalesce("discount_amount", lit(0)).cast("decimal(12,2)"))
    .withColumn("net_item_revenue", coalesce("net_item_revenue", lit(0)).cast("decimal(12,2)"))
    .withColumn("delivery_fee_revenue", coalesce("delivery_fee_revenue", lit(0)).cast("decimal(12,2)"))
    .withColumn("on_time_orders", coalesce("on_time_orders", lit(0)).cast("int"))
    .withColumn(
        "on_time_rate_pct",
        when(col("delivered_orders") > 0,
             round(lit(100) * col("on_time_orders") / col("delivered_orders"), 2)).cast("decimal(5,2)")
    )
    .select(
        col("order_date").cast("date"),
        "store_id", "city", "delivered_orders", "cancelled_orders", "units_sold",
        "gross_amount", "discount_amount", "net_item_revenue", "delivery_fee_revenue",
        "on_time_orders", "on_time_rate_pct",
    )
)

(agg_daily_store_sales.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.agg_daily_store_sales"))

In [0]:
from pyspark.sql.functions import *

fact = spark.read.table(f"{catalog}.gold.fact_orders")

delivered = col("order_status") == "DELIVERED"

order_kpis = fact.agg(
    count(when(delivered, 1)).alias("delivered_orders"),
    count(when(col("order_status") == "CANCELLED", 1)).alias("cancelled_orders"),
    coalesce(sum(when(delivered, col("net_item_amount"))), lit(0)).alias("net_item_revenue"),
    count(when(delivered & (col("is_on_time") == True), 1)).alias("on_time_orders"),
)

customer_counts = (
    fact.filter(delivered & col("customer_id").isNotNull())
    .groupBy("customer_id")
    .agg(count("*").alias("delivered_cnt"))
)

customer_kpis = customer_counts.agg(
    count("*").alias("active_customers"),
    count(when(col("delivered_cnt") >= 2, 1)).alias("repeat_customers"),
)

kpi_summary = (
    order_kpis.crossJoin(customer_kpis)
    .select(
        col("delivered_orders").cast("int"),
        col("cancelled_orders").cast("int"),
        col("net_item_revenue").cast("decimal(12,2)"),
        (col("net_item_revenue") / when(col("delivered_orders") > 0, col("delivered_orders")))
            .cast("decimal(12,2)").alias("avg_order_value"),
        (100 * col("on_time_orders") / when(col("delivered_orders") > 0, col("delivered_orders")))
            .cast("decimal(5,2)").alias("on_time_rate_pct"),
        (100 * col("cancelled_orders") /
            when((col("delivered_orders") + col("cancelled_orders")) > 0,
                 col("delivered_orders") + col("cancelled_orders")))
            .cast("decimal(5,2)").alias("cancellation_rate_pct"),
        col("active_customers").cast("int"),
        (100 * col("repeat_customers") / when(col("active_customers") > 0, col("active_customers")))
            .cast("decimal(5,2)").alias("repeat_customer_rate_pct"),
    )
)

(kpi_summary.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.kpi_summary"))

spark.table(f"{catalog}.gold.kpi_summary").display()

delivered_orders,cancelled_orders,net_item_revenue,avg_order_value,on_time_rate_pct,cancellation_rate_pct,active_customers,repeat_customer_rate_pct
1017,68,496540.00,488.24,87.32,6.27,351,67.52
